# Streaming a Response with InvokeModelWithResponseStream

Same model and payload as the InvokeModel demo, but instead of waiting for the whole answer, we read it in chunks as the model generates it. This is what powers the "typing" effect you see in chat apps.

Steps:
1. Build the payload (same as before)
2. Call `invoke_model_with_response_stream` and print text as it arrives
3. Look at the raw event chunks that make up the stream

## 1. Build the payload

In [ ]:
import boto3
import json

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 5000,
    "messages": [
        {
            "role": "user",
            "content": "Create a script to resize images"
        }
    ]
})

## 2. Stream the response

The response body is an iterator of events. Each `content_block_delta` chunk carries a small piece of text. We print each piece as it arrives (`end=""` keeps it on one flowing line).

In [ ]:
response = bedrock_runtime.invoke_model_with_response_stream(
    body=body,
    modelId="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    accept="application/json",
    contentType="application/json"
)

print("Streaming invoke response:")
stream = response.get("body")
for event in stream:
    chunk = event.get("chunk")
    if chunk:
        chunk_obj = json.loads(chunk.get("bytes").decode())
        if chunk_obj["type"] == "content_block_delta":
            print(chunk_obj["delta"]["text"], end="", flush=True)

## 3. The event chunks

A stream is a sequence of typed events. To see the structure, we make the call again and collect the `type` of every chunk. You'll see it start with `message_start`, stream many `content_block_delta` chunks (the actual text), and end with `message_stop`.

In [ ]:
response = bedrock_runtime.invoke_model_with_response_stream(
    body=body,
    modelId="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    accept="application/json",
    contentType="application/json"
)

chunk_types = []
first_of_each = {}
for event in response.get("body"):
    chunk = event.get("chunk")
    if chunk:
        obj = json.loads(chunk.get("bytes").decode())
        chunk_types.append(obj["type"])
        # Keep the first example of each chunk type to inspect
        first_of_each.setdefault(obj["type"], obj)

print("Chunk types in order:")
for t in chunk_types:
    print(" ", t)

One example of each chunk type, so you can see what's inside them:

In [ ]:
for chunk_type, example in first_of_each.items():
    print(f"=== {chunk_type} ===")
    print(json.dumps(example, indent=2))
    print()